In [22]:
# Final ML summary

final_ml_summary = pd.DataFrame([
    linear_result,
    gb_result,
    rf_result,
    realistic_result
])

final_ml_summary = final_ml_summary.sort_values(by="R2", ascending=False)

final_ml_summary_path = PROCESSED_DATA_DIR / "ml_final_model_summary.csv"
final_ml_summary.to_csv(final_ml_summary_path, index=False)

final_ml_summary

,model_name,MAE,RMSE,R2
2,Random Forest Regressor,"35,637.92","395,076.73",1.00
1,Gradient Boosting Regressor,"257,726.47","810,953.44",0.98
3,Realistic Random Forest Without Price Leakage,"1,090,940.61","3,756,993.52",0.62
0,Linear Regression,"2,168,947.29","5,709,399.21",0.12


In [21]:
# Save realistic actual vs predicted results

realistic_prediction_results = pd.DataFrame({
    "actual_price": y_test_r.values,
    "predicted_price": realistic_predictions
})

realistic_prediction_results["prediction_error"] = (
    realistic_prediction_results["actual_price"] - realistic_prediction_results["predicted_price"]
)

realistic_prediction_results["absolute_error"] = realistic_prediction_results["prediction_error"].abs()

realistic_prediction_path = PROCESSED_DATA_DIR / "ml_realistic_actual_vs_predicted.csv"
realistic_prediction_results.to_csv(realistic_prediction_path, index=False)

print("Realistic prediction results saved to:", realistic_prediction_path)

realistic_prediction_results.head()

Realistic prediction results saved to: c:\Projects\dubai-real-estate-price-intelligence-dashboard\data\processed\ml_realistic_actual_vs_predicted.csv


,actual_price,predicted_price,prediction_error,absolute_error
0,"1,325,000.00","1,619,442.61","-294,442.61","294,442.61"
1,"3,300,000.00","3,386,465.54","-86,465.54","86,465.54"
2,"714,888.00","471,408.22","243,479.78","243,479.78"
3,"1,578,000.00","920,717.75","657,282.25","657,282.25"
4,"1,294,643.00","857,021.05","437,621.95","437,621.95"


In [20]:
# Save realistic model

realistic_model_path = MODELS_DIR / "dubai_real_estate_realistic_price_model.pkl"

joblib.dump(realistic_rf_model, realistic_model_path)

print("Realistic model saved to:", realistic_model_path)

Realistic model saved to: c:\Projects\dubai-real-estate-price-intelligence-dashboard\models\dubai_real_estate_realistic_price_model.pkl


In [19]:
# Build realistic model without meter_sale_price
# meter_sale_price is highly related to actual_worth, so we remove it for a more realistic model

realistic_features = [
    "property_type_en",
    "property_sub_type_en",
    "property_usage_en",
    "reg_type_en",
    "area_name_en",
    "rooms_en",
    "has_parking",
    "procedure_area",
    "transaction_year",
    "transaction_month",
    "transaction_quarter"
]

realistic_features = [col for col in realistic_features if col in df_model.columns]

X_realistic = df_model[realistic_features]
y_realistic = df_model["actual_worth"]

categorical_features_realistic = X_realistic.select_dtypes(include="object").columns.tolist()
numeric_features_realistic = X_realistic.select_dtypes(include=["int64", "float64"]).columns.tolist()

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_realistic,
    y_realistic,
    test_size=0.2,
    random_state=42
)

preprocessor_realistic = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features_realistic),
        ("num", "passthrough", numeric_features_realistic)
    ]
)

realistic_rf_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor_realistic),
        ("model", RandomForestRegressor(
            n_estimators=100,
            max_depth=15,
            random_state=42,
            n_jobs=-1
        ))
    ]
)

realistic_rf_model.fit(X_train_r, y_train_r)

realistic_result, realistic_predictions = evaluate_model(
    "Realistic Random Forest Without Price Leakage",
    realistic_rf_model,
    X_test_r,
    y_test_r
)

realistic_result

C:\Users\hanir\AppData\Local\Temp\ipykernel_23388\3841204507.py:23: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features_realistic = X_realistic.select_dtypes(include="object").columns.tolist()


{'model_name': 'Realistic Random Forest Without Price Leakage',
 'MAE': 1090940.612621235,
 'RMSE': np.float64(3756993.515072817),
 'R2': 0.6174103284017736}

In [18]:
# Quick prediction test using one sample row

sample_input = X_test.iloc[[0]]
actual_value = y_test.iloc[0]
predicted_value = rf_model.predict(sample_input)[0]

print("Actual price:", round(actual_value, 2))
print("Predicted price:", round(predicted_value, 2))
print("Difference:", round(actual_value - predicted_value, 2))

sample_input

Actual price: 1325000.0
Predicted price: 1315627.07
Difference: 9372.93


,property_type_en,property_sub_type_en,property_usage_en,reg_type_en,area_name_en,rooms_en,has_parking,procedure_area,meter_sale_price,transaction_year,transaction_month,transaction_quarter
271515,Unit,Flat,Residential,Existing Properties,Al Thanyah Fifth,1 B/R,1,138.68,"9,554.37","2,022.00",12.00,4.00


In [17]:
# Save best model

best_model_path = MODELS_DIR / "dubai_real_estate_price_prediction_model.pkl"

joblib.dump(rf_model, best_model_path)

print("Best model saved to:", best_model_path)

Best model saved to: c:\Projects\dubai-real-estate-price-intelligence-dashboard\models\dubai_real_estate_price_prediction_model.pkl


In [16]:
# Save actual vs predicted results

prediction_results_path = PROCESSED_DATA_DIR / "ml_actual_vs_predicted.csv"
prediction_results.to_csv(prediction_results_path, index=False)

print("Prediction results saved to:", prediction_results_path)

Prediction results saved to: c:\Projects\dubai-real-estate-price-intelligence-dashboard\data\processed\ml_actual_vs_predicted.csv


In [15]:
# Create actual vs predicted comparison for best model

best_predictions = rf_predictions

prediction_results = pd.DataFrame({
    "actual_price": y_test.values,
    "predicted_price": best_predictions
})

prediction_results["prediction_error"] = prediction_results["actual_price"] - prediction_results["predicted_price"]
prediction_results["absolute_error"] = prediction_results["prediction_error"].abs()

prediction_results.head()

,actual_price,predicted_price,prediction_error,absolute_error
0,"1,325,000.00","1,315,627.07","9,372.93","9,372.93"
1,"3,300,000.00","3,307,335.78","-7,335.78","7,335.78"
2,"714,888.00","720,093.01","-5,205.01","5,205.01"
3,"1,578,000.00","1,583,811.78","-5,811.78","5,811.78"
4,"1,294,643.00","1,295,830.53","-1,187.53","1,187.53"


In [14]:
# Save model comparison results

model_results_path = PROCESSED_DATA_DIR / "ml_model_comparison.csv"
model_results.to_csv(model_results_path, index=False)

print("Model comparison saved to:", model_results_path)

Model comparison saved to: c:\Projects\dubai-real-estate-price-intelligence-dashboard\data\processed\ml_model_comparison.csv


In [13]:
# Model comparison table

model_results = pd.DataFrame([
    linear_result,
    rf_result,
    gb_result
])

model_results = model_results.sort_values(by="R2", ascending=False)

model_results

,model_name,MAE,RMSE,R2
1,Random Forest Regressor,"35,637.92","395,076.73",1.00
2,Gradient Boosting Regressor,"257,726.47","810,953.44",0.98
0,Linear Regression,"2,168,947.29","5,709,399.21",0.12


In [12]:
# Gradient Boosting model

gb_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", GradientBoostingRegressor(
            n_estimators=100,
            learning_rate=0.1,
            max_depth=3,
            random_state=42
        ))
    ]
)

gb_model.fit(X_train, y_train)

gb_result, gb_predictions = evaluate_model(
    "Gradient Boosting Regressor",
    gb_model,
    X_test,
    y_test
)

gb_result

{'model_name': 'Gradient Boosting Regressor',
 'MAE': 257726.47493554567,
 'RMSE': np.float64(810953.4411670024),
 'R2': 0.9821743985264982}

In [11]:
# Random Forest model

rf_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestRegressor(
            n_estimators=100,
            max_depth=15,
            random_state=42,
            n_jobs=-1
        ))
    ]
)

rf_model.fit(X_train, y_train)

rf_result, rf_predictions = evaluate_model(
    "Random Forest Regressor",
    rf_model,
    X_test,
    y_test
)

rf_result

{'model_name': 'Random Forest Regressor',
 'MAE': 35637.920057232775,
 'RMSE': np.float64(395076.732090436),
 'R2': 0.9957692705230693}

In [10]:
# Linear Regression model

linear_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LinearRegression())
    ]
)

linear_model.fit(X_train, y_train)

linear_result, linear_predictions = evaluate_model(
    "Linear Regression",
    linear_model,
    X_test,
    y_test
)

linear_result

{'model_name': 'Linear Regression',
 'MAE': 2168947.290895471,
 'RMSE': np.float64(5709399.208883545),
 'R2': 0.11644584850831774}

In [9]:
# Helper function to evaluate models

def evaluate_model(model_name, model, X_test, y_test):
    predictions = model.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    r2 = r2_score(y_test, predictions)

    result = {
        "model_name": model_name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    }

    return result, predictions

In [8]:
# Preprocessing pipeline

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", "passthrough", numeric_features)
    ]
)

print("Preprocessor created.")

Preprocessor created.


In [7]:
# Train test split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (120000, 12)
X_test shape: (30000, 12)
y_train shape: (120000,)
y_test shape: (30000,)


In [6]:
# Define features and target

target = "actual_worth"

X = df_model.drop(columns=[target])
y = df_model[target]

categorical_features = X.select_dtypes(include="object").columns.tolist()
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

print("Target:", target)
print("Categorical features:", categorical_features)
print("Numeric features:", numeric_features)

Target: actual_worth
Categorical features: ['property_type_en', 'property_sub_type_en', 'property_usage_en', 'reg_type_en', 'area_name_en', 'rooms_en']
Numeric features: ['has_parking', 'procedure_area', 'meter_sale_price', 'transaction_year', 'transaction_month', 'transaction_quarter']


C:\Users\hanir\AppData\Local\Temp\ipykernel_23388\2619330312.py:8: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(include="object").columns.tolist()


In [5]:
# Use sample for faster training

sample_size = 150000

if len(df_ml) > sample_size:
    df_model = df_ml.sample(n=sample_size, random_state=42)
else:
    df_model = df_ml.copy()

print("Model training dataset shape:", df_model.shape)

Model training dataset shape: (150000, 13)


In [4]:
# Remove missing and invalid values

before_rows = len(df_ml)

df_ml = df_ml.dropna()

df_ml = df_ml[
    (df_ml["actual_worth"] > 0) &
    (df_ml["procedure_area"] > 0) &
    (df_ml["meter_sale_price"] > 0)
].copy()

after_rows = len(df_ml)

print("Rows before cleaning:", before_rows)
print("Rows after cleaning:", after_rows)
print("Rows removed:", before_rows - after_rows)

Rows before cleaning: 1044151
Rows after cleaning: 1044151
Rows removed: 0


In [3]:
# Select columns for ML model

ml_columns = [
    "actual_worth",
    "property_type_en",
    "property_sub_type_en",
    "property_usage_en",
    "reg_type_en",
    "area_name_en",
    "rooms_en",
    "has_parking",
    "procedure_area",
    "meter_sale_price",
    "transaction_year",
    "transaction_month",
    "transaction_quarter"
]

ml_columns = [col for col in ml_columns if col in df.columns]

df_ml = df[ml_columns].copy()

print("ML dataset shape:", df_ml.shape)
df_ml.head()

ML dataset shape: (1044151, 13)


,actual_worth,property_type_en,property_sub_type_en,property_usage_en,reg_type_en,area_name_en,rooms_en,has_parking,procedure_area,meter_sale_price,transaction_year,transaction_month,transaction_quarter
0,"1,350,000.00",Land,Unknown,Commercial,Existing Properties,Al Wasl,Unknown,0,"1,393.55",968.75,"2,001.00",2.00,1.00
1,"2,790,000.00",Villa,Unknown,Commercial,Existing Properties,Al Hudaiba,Unknown,0,"1,728.00","1,614.58","2,004.00",12.00,4.00
2,"20,000,000.00",Land,Unknown,Commercial,Existing Properties,Burj Khalifa,Unknown,0,929.03,"21,527.83","2,001.00",3.00,1.00
3,"25,000,000.00",Building,Unknown,Residential / Commercial,Existing Properties,Oud Metha,Unknown,0,"2,673.28","9,351.81","2,005.00",9.00,3.00
4,"9,000,000.00",Villa,Unknown,Residential,Existing Properties,Al Bada,Unknown,0,"1,541.17","5,839.72","2,012.00",10.00,4.00


In [2]:
# Load cleaned dataset

file_path = PROCESSED_DATA_DIR / "dubai_real_estate_transactions_cleaned.csv"

df = pd.read_csv(file_path, low_memory=False)

print("Dataset loaded successfully.")
print("Shape:", df.shape)

df.head()

Dataset loaded successfully.
Shape: (1044151, 27)


,transaction_id,procedure_id,trans_group_en,procedure_name_en,instance_date,property_type_en,property_sub_type_en,property_usage_en,reg_type_en,area_id,area_name_en,building_name_en,project_name_en,master_project_en,nearest_metro_en,nearest_mall_en,nearest_landmark_en,rooms_en,has_parking,procedure_area,actual_worth,meter_sale_price,meter_rent_price,transaction_year,transaction_month,transaction_quarter,transaction_year_month
0,1-11-2001-165,11,Sales,Sell,2001-02-24,Land,Unknown,Commercial,Existing Properties,364,Al Wasl,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,0,"1,393.55","1,350,000.00",968.75,NaN,"2,001.00",2.00,1.00,2001-02
1,3-9-2004-223,9,Gifts,Grant,2004-12-13,Villa,Unknown,Commercial,Existing Properties,365,Al Hudaiba,Unknown,Unknown,Unknown,Al Jafiliya Metro Station,Dubai Mall,Burj Khalifa,Unknown,0,"1,728.00","2,790,000.00","1,614.58",NaN,"2,004.00",12.00,4.00,2004-12
2,2-13-1996-119,13,Mortgages,Mortgage Registration,2001-03-12,Land,Unknown,Commercial,Existing Properties,390,Burj Khalifa,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,0,929.03,"20,000,000.00","21,527.83",NaN,"2,001.00",3.00,1.00,2001-03
3,2-14-2005-222,14,Mortgages,Modify Mortgage,2005-09-20,Building,Unknown,Residential / Commercial,Existing Properties,388,Oud Metha,Unknown,Unknown,Unknown,Oud Metha Metro Station,Dubai Mall,Dubai International Airport,Unknown,0,"2,673.28","25,000,000.00","9,351.81",NaN,"2,005.00",9.00,3.00,2005-09
4,3-9-2012-874,9,Gifts,Grant,2012-10-11,Villa,Unknown,Residential,Existing Properties,276,Al Bada,Unknown,Unknown,Unknown,Trade Centre Metro Station,Dubai Mall,Burj Khalifa,Unknown,0,"1,541.17","9,000,000.00","5,839.72",NaN,"2,012.00",10.00,4.00,2012-10


In [1]:
# Dubai Real Estate Price Intelligence Dashboard
# Notebook 05: Machine Learning Price Prediction

import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression

import joblib

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:,.2f}".format)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"

print("Setup completed.")
print("Processed data folder:", PROCESSED_DATA_DIR)
print("Models folder:", MODELS_DIR)

Setup completed.
Processed data folder: c:\Projects\dubai-real-estate-price-intelligence-dashboard\data\processed
Models folder: c:\Projects\dubai-real-estate-price-intelligence-dashboard\models
